In [1]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout

In [2]:
train_data_dir = 'imagedataset\dataset'

In [3]:
train_datagen = ImageDataGenerator(rescale = 1./255.,rotation_range = 40, width_shift_range = 0.2, height_shift_range = 0.2, shear_range = 0.2, zoom_range = 0.2, horizontal_flip = True)

In [4]:
train_generator = train_datagen.flow_from_directory(train_data_dir, batch_size = 20, class_mode = 'binary', target_size = (224, 224))

Found 2829 images belonging to 2 classes.


In [5]:
from tensorflow.keras.applications import VGG19
from tensorflow.keras.callbacks import EarlyStopping
import tensorflow as tf

# Load ResNet50 model
base_model = VGG19(input_shape=(224, 224, 3), include_top=False, weights='imagenet')

# Freeze the layers in the base model
for layer in base_model.layers:
    layer.trainable = False

# Add your custom fully connected layers
x = tf.keras.layers.GlobalAveragePooling2D()(base_model.output)
x = tf.keras.layers.Dense(512, activation='relu')(x)
x = tf.keras.layers.Dropout(0.5)(x)
x = tf.keras.layers.Dense(1, activation='sigmoid')(x)

# Create the new model
model = tf.keras.models.Model(base_model.input, x)

# Add EarlyStopping callback to monitor training loss
early_stopping = EarlyStopping(monitor='loss', patience=3, restore_best_weights=True)

# Compile the model with the callback
model.compile(optimizer=tf.keras.optimizers.RMSprop(learning_rate=0.0001), 
              loss='binary_crossentropy', 
              metrics=['acc'])

# Train the model with early stopping using generator
vg16_hist = model.fit(train_generator, 
                        steps_per_epoch=20, 
                        epochs=10, 
                        callbacks=[early_stopping])

# Print the final accuracy
print("Final Accuracy: ", vg16_hist.history['acc'][-1])


Epoch 1/10
20/20 [==============================] - 23s 271ms/step - loss: 0.5526 - acc: 0.7525
Epoch 2/10
20/20 [==============================] - 6s 293ms/step - loss: 0.5564 - acc: 0.7825
Epoch 3/10
20/20 [==============================] - 6s 290ms/step - loss: 0.5823 - acc: 0.7600
Epoch 4/10
20/20 [==============================] - 9s 454ms/step - loss: 0.4958 - acc: 0.8175
Epoch 5/10
20/20 [==============================] - 5s 255ms/step - loss: 0.5438 - acc: 0.7925
Epoch 6/10
20/20 [==============================] - 5s 249ms/step - loss: 0.5147 - acc: 0.7975
Epoch 7/10
20/20 [==============================] - 5s 259ms/step - loss: 0.5682 - acc: 0.7650
Final Accuracy:  0.7649999856948853


In [5]:
from tensorflow.keras.applications import VGG19
from tensorflow.keras.callbacks import EarlyStopping
import tensorflow as tf
base_model = VGG19(input_shape=(224, 224, 3), include_top=False, weights='imagenet')

for layer in base_model.layers[:100]:
    layer.trainable = False
for layer in base_model.layers[100:]:
    layer.trainable = True

x = tf.keras.layers.GlobalAveragePooling2D()(base_model.output)
x = tf.keras.layers.Dense(1024, activation='relu')(x)
x = tf.keras.layers.Dropout(0.5)(x)
x = tf.keras.layers.Dense(512, activation='relu')(x)
x = tf.keras.layers.Dropout(0.5)(x)
x = tf.keras.layers.Dense(1, activation='sigmoid')(x)

model = tf.keras.models.Model(base_model.input, x)

early_stopping = EarlyStopping(monitor='acc', patience=5, restore_best_weights=True)

model.compile(optimizer=tf.keras.optimizers.RMSprop(lr=0.0001), 
              loss='binary_crossentropy', 
              metrics=['acc'])
inception_hist = model.fit(train_generator, 
                           steps_per_epoch=len(train_generator),
                           epochs=15,
                           callbacks=[early_stopping])

print("Final Accuracy: ", inception_hist.history['acc'][-1])

c:\Users\abhinav bhardwaj\miniconda3\envs\tl\lib\site-packages\keras\optimizers\optimizer_v2\rmsprop.py:140: UserWarning: The `lr` argument is deprecated, use `learning_rate` instead.
  super().__init__(name, **kwargs)


Epoch 1/15
142/142 [==============================] - 61s 339ms/step - loss: 0.5282 - acc: 0.7897
Epoch 2/15
142/142 [==============================] - 39s 275ms/step - loss: 0.5185 - acc: 0.7939
Epoch 3/15
142/142 [==============================] - 36s 252ms/step - loss: 0.5078 - acc: 0.7957
Epoch 4/15
142/142 [==============================] - 37s 257ms/step - loss: 0.5030 - acc: 0.7964
Epoch 5/15
142/142 [==============================] - 36s 255ms/step - loss: 0.5009 - acc: 0.7964
Epoch 6/15
142/142 [==============================] - 36s 253ms/step - loss: 0.4979 - acc: 0.7975
Epoch 7/15
142/142 [==============================] - 36s 252ms/step - loss: 0.4986 - acc: 0.7964
Epoch 8/15
142/142 [==============================] - 36s 251ms/step - loss: 0.4945 - acc: 0.7967
Epoch 9/15
142/142 [==============================] - 38s 265ms/step - loss: 0.4861 - acc: 0.7957
Epoch 10/15
142/142 [==============================] - 38s 263ms/step - loss: 0.4825 - acc: 0.7964
Epoch 11/15
142/142

In [6]:
model.save('vg19.h5')